In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [2]:
def solve_qbd_full_states(L1, L2, R1, c1, R2, c2, mu):
    # TYPE 1
    h1 = ((R1/c1)*mu**2 - L2) / ( L1 + (R1/c1)*mu*L1)
    if h1 < 0:
        h1 = 0
    if h1 > 1:
        h1 = 1

    #TYPE 2
    n_star = math.floor((mu-h1*L1)*R2/c2)
    lam1 = L1 * h1
    if not (lam1 < mu):
        raise ValueError("Stability requires L1*h1 < mu.")
        
    dim = n_star + 1
    I = np.eye(dim)
    
    A1 = lam1 * I
    Am1 = mu * I
    A0 = -(lam1 + mu) * I  # for i >= n*

    R = np.zeros((dim, dim))
    A0_inv = np.linalg.inv(A0)
    for _ in range(2000):
        R_next = -(A1 + (R @ R) @ Am1) @ A0_inv
        if np.linalg.norm(R_next - R, ord=np.inf) < 1e-14:
            R = R_next
            break
        R = R_next

    # level dependent Ri 
    R_list = [None] * (n_star + 1)
    R_list[n_star] = R
    for i in range(n_star - 1, 0, -1):
        A0_n = np.zeros((dim, dim))
        for j in range(dim):
            rate2 = L2 if j < (n_star - i) else 0
            A0_n[j, j] = -(lam1 + rate2 + mu)
            if j < dim - 1:
                A0_n[j, j+1] = rate2
        R_list[i] = -A1 @ np.linalg.inv(A0_n + R_list[i+1] @ Am1)

    sigma = -(lam1 + L2)
    beta = np.zeros((1, dim)); beta[0, 0] = (lam1 + L2)
    b = np.zeros((dim, 1)); b[0, 0] = mu
    B0 = np.zeros((dim, dim))
    for j in range(dim):
        rate2 = L2 if j < n_star else 0
        B0[j, j] = -(lam1 + rate2 + mu)
        if j < n_star: B0[j, j+1] = rate2
        if j > 0: B0[j, j-1] = mu
    B0[0,0] = -(lam1 + L2 + mu)

    if n_star >= 1:
        K = B0 + R_list[1] @ Am1 - (1/sigma) * (b @ beta)
    else:
        K = B0 + R @ Am1 - (1/sigma) * (b @ beta)
    vals, vecs = np.linalg.eig(K.T)
    pi_0 = np.real(vecs[:, np.argmin(np.abs(vals))])
    pi_0 = pi_0 / np.sum(pi_0)

    pi_levels = [None] * (n_star + 1)
    pi_levels[0] = pi_0
    for i in range(1, n_star + 1):
        pi_levels[i] = pi_levels[i-1] @ R_list[i]

    pi_empty_val = -float(np.dot(pi_0, b)) / sigma
    
    total = pi_empty_val + sum(np.sum(p) for p in pi_levels[:n_star])
    total += np.sum(pi_levels[n_star] @ np.linalg.inv(I - R))
    
    pi_empty_final = pi_empty_val / total
    pi_final = [p / total for p in pi_levels]

    p_join2 = pi_empty_final
    pi000 = float(pi_empty_final)
    results = {(0,0,0):pi000}

    for i in range(n_star + 1):
        for j in range(dim):
            prob = pi_final[i][j]
            if i+j+1 <= n_star:
                p_join2 += prob
                results[(i,j,1)] = float(prob)
            

    h2 = float(p_join2)
    

    return h1, h2, results


#solve_qbd_full_states(1, 1, 10, 4, 8, 2, 2.1)

In [3]:
def metrics(L1, L2, R1, c1, R2, c2, mu):
    h1, h2, pi_states = solve_qbd_full_states(L1, L2, R1, c1, R2, c2, mu)
    m_star = math.floor((mu-h1*L1)*R2/c2)
    state = list(pi_states.keys())

    ### W1
    W1 = (L1*h1 + L2)/(mu*(mu-L1*h1))

    ### U1
    U1 = (R1-c1*W1)

    ### W2
    W2 = 0
    for s in state:
        i, j, k = s[0], s[1], s[2]
        if i + j + k <= m_star:
            W2 += pi_states[s]*(i+j+k)/(mu - L1*h1)


    ### U2 
    U2 = 0
    for s in state:
        i, j, k = s[0], s[1], s[2]
        if i + j + k <= m_star:
            U2 += pi_states[s]*( R2 - c2*( (i+j+k)/(mu - L1*h1)  )  )

    ### throughput
    thr = (L1*h1 + L2*h2)/(L1 + L2)

    ### cumulative utility
    cumU = L1*h1*U1 + L2*h2*U2

    dic = {(L1, L2, R1, c1, R2, c2, mu): (h1, h2, W1, W2, U1, U2, thr, cumU) }

    return dic
    